In [44]:
# 启用自动重载
%load_ext autoreload
%autoreload 2

import logging

from pulao.logging import init_logging

import pandas as pd

from pulao.bar import SBarManager,SBar,CBarManager
from pulao.swing import SwingManager

import polars as pl
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from pulao.constant import Exchange, FractalType, Timeframe,Direction

from pulao.logging import get_logger
init_logging(level=logging.DEBUG)
open("./logs/pulao.log", "w").close() # 每次运行前清空日志
logger = get_logger(__name__)
logger.debug("开始运行")

df = pl.read_csv("../dataset/I8888.XDCE_60m.csv", try_parse_dates=True)
df = df[280:330]  # test

sbar_list = []
columns = df.columns

symbol="i8888"
exchange = Exchange.DCE
tieframe = Timeframe.H1
for idx, row in enumerate(df.iter_rows()):
    row_dict = dict(zip(columns, row, strict=False))
    # datetime,open,close,high,low,volume,money,open_interest,signal
    datetime = row_dict["datetime"]
    o = row_dict["open"]
    close = row_dict["close"]
    high = row_dict["high"]
    low = row_dict["low"]
    volume = row_dict["volume"]
    money = row_dict["money"]
    open_interest = row_dict["open_interest"]

    sbar = SBar(id=idx, symbol=symbol, exchange=exchange, timeframe=tieframe, datetime=datetime, turnover=money, open_price=o, close_price=close, high_price=high, low_price=low, volume=volume,open_interest=open_interest, ema_short=0, ema_long=0, created_at=datetime)
    sbar_list.append(sbar)
# 模拟行情数据接收
sbar_manager = SBarManager(symbol=symbol, timeframe=tieframe)
cbar_manager = CBarManager(sbar_manager=sbar_manager)
swing_manager = SwingManager(cbar_manager=cbar_manager)

# 流入模拟数据
for sbar in sbar_list:
    sbar_manager.append(sbar)

logger.debug("运行结束")
# sleep(3) # 等一会，让日志输出完

#
# Plotly 绘图
#
# 设置全局默认布局
pio.templates.default = 'plotly_dark'
#
# region . Plotly - Cbar
#
df_cbar = cbar_manager.df_cbar.to_pandas()
df_cbar["datetime"] = pd.date_range("2025-01-01", periods=df_cbar.shape[0], freq="h")
df_cbar["open_price"] = df_cbar["high_price"]
df_cbar["close_price"] = df_cbar["low_price"]
df_cbar.insert(0, "index", range(len(df_cbar)))

# region 构建波段数据
df_swing = swing_manager.df_swing.to_pandas()
df_swing["start_datetime"] = pd.Series(dtype="datetime64[ns]")
df_swing["end_datetime"] = pd.Series(dtype="datetime64[ns]")
df_swing.insert(0, "index", range(len(df_swing)))

for i, row in enumerate(df_swing.itertuples()):
    df = df_cbar[(df_cbar["id"] >= row.cbar_start_id) & (df_cbar["id"] <= row.cbar_end_id)]
    start_datetime = df.iloc[0]["datetime"]
    end_datetime = df.iloc[-1]["datetime"]
    df_swing.at[i, "start_datetime"] = start_datetime
    df_swing.at[i, "end_datetime"] = end_datetime

xs = []
ys = []
texts = []
text_positions  = []
for _, row in enumerate(df_swing.itertuples()):
    xs += [row.start_datetime, row.end_datetime, None]
    start_index = df_cbar[(df_cbar["id"] == row.cbar_start_id)]["index"].iloc[0]
    end_index = df_cbar[(df_cbar["id"] == row.cbar_end_id)]["index"].iloc[0]
    if row.direction == Direction.UP:
        start_price = df_cbar[(df_cbar["id"] == row.cbar_start_id)]["low_price"].iloc[0]
        end_price = df_cbar[(df_cbar["id"] == row.cbar_end_id)]["high_price"].iloc[0]
        text_positions += ['bottom center', 'top center', "middle center"]
    else:
        start_price = df_cbar[(df_cbar["id"] == row.cbar_start_id)]["high_price"].iloc[0]
        end_price = df_cbar[(df_cbar["id"] == row.cbar_end_id)]["low_price"].iloc[0]
        text_positions += ['top center', 'bottom center', "middle center"]
    ys += [start_price, end_price, None]
    texts += [start_index, end_index, None]


# endregion

fig = make_subplots(
    rows=1, cols=1
)
fig.add_trace(go.Candlestick(
    x=df_cbar['datetime'],
    open=df_cbar['open_price'],
    high=df_cbar['high_price'],
    low=df_cbar['low_price'],
    close=df_cbar['close_price'],
    name='K线',
), row=1, col=1)

df_fractal_bottom = df_cbar[(df_cbar['fractal_type'] == FractalType.BOTTOM)]

fig.add_trace(go.Scatter(
    x=df_fractal_bottom['datetime'],
    y=df_fractal_bottom['low_price'] * 0.995,   # 放在K线最低价下方一点，不遮挡蜡烛
    mode='markers+text',
    name='swing point low',
    marker=dict(
        symbol='triangle-up',
        size=14,
        color='#1E90FF',
    ),
    text=df_fractal_bottom["index"],      # ← 添加序号
    textposition="bottom center",              # ← 文字位置
    textfont=dict(size=10, color="white"),
    hovertemplate=
        "<b>波段低点</b><br>" +
        "时间: %{x}<br>" +
        "价格: %{y:.2f}<br>" +
        "<extra></extra>"
), row=1, col=1)


df_fractal_top = df_cbar[(df_cbar['fractal_type'] == FractalType.TOP)]

fig.add_trace(go.Scatter(
    x=df_fractal_top['datetime'],
    y=df_fractal_top['high_price'] * 1.005,  # 放在K线最高价上方一点
    mode='markers+text',
    name='swing point high',
    marker=dict(
        symbol='triangle-down',
        size=14,
        color='#FF4500',
    ),
    text=df_fractal_top["index"],      # ← 添加序号
    textposition="top center",              # ← 文字位置
    textfont=dict(size=10, color="white"),
    hovertemplate=
        "<b>波段高点</b><br>" +
        "时间: %{x}<br>" +
        "价格: %{y:.2f}<br>" +
        "<extra></extra>"
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=xs,
    y=ys,
    name='swing segments',
    mode='lines+text',    # lines+text 支持显示文字
    line=dict(width=2, color='orange'),
    text=texts,
    textposition=text_positions,  # 文字位置
))

title = 'Swing Chart - 波段标识'
fig.update_layout(title=title,
                  height=900,
                  hovermode='x unified',  # X轴悬停联动虚线
                  )

fig.update_xaxes(
    showgrid=False,
)

fig.update_yaxes(
    showgrid=False,
)

fig.show()

# endregion



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
{"event": "开始运行", "level": "debug", "logger": "__main__", "time": "2026-03-28 02:54:47"}
{"cbar_count": 1, "event": "用于组成分形的cbar数量不够", "level": "info", "logger": "pulao.swing.swing_manager", "time": "2026-03-28 02:54:47"}
{"cbar_count": 2, "event": "用于组成分形的cbar数量不够", "level": "info", "logger": "pulao.swing.swing_manager", "time": "2026-03-28 02:54:47"}
{"drop_fractal": "Fractal(left=CBar(id=163480916721668096, sbar_start_id=163480916700692480, sbar_end_id=163480916717469696, high_price=783.6019897460938, low_price=777.1929931640625, created_at=datetime.datetime(2026, 3, 28, 2, 54, 47, 875000), fractal_type=0), middle=CBar(id=163480916767805440, sbar_start_id=163480916763607040, sbar_end_id=163480916763607040, high_price=780.572021484375, low_price=773.1380004882812, created_at=datetime.datetime(2026, 3, 28, 2, 54, 47, 886000), fractal_type=0), right=CBar(id=163480916784582656, sbar_start_id=16348091

In [35]:
# swing = swing_manager.get_last_swing()
# print(f"swing: {swing}")
# idx = swing_manager.get_idx(swing.id) if swing else None
# print(f"idx: {idx}")
# prev_swing = swing_manager.get_swing_by_idx(idx-1) if idx is not None else None
# print(f"prev_swing: {prev_swing}")

# idx = swing_manager.df_swing.select(pl.col("id").search_sorted(swing.id)).item() if swing else None
# print(f"idx in df_swing: {idx}")

display(swing_manager.df_swing)

id,cbar_start_id,cbar_end_id,sbar_start_id,sbar_end_id,high_price,low_price,direction,span,volume,start_oi,end_oi,state,created_at
u64,u64,u64,u64,u64,f32,f32,i8,u32,f32,f32,f32,i8,datetime[ms]
163475734231269376,163475733941850112,163475734680047616,163475733950234624,163475734705209344,791.991028,737.958984,1,24,1.728105e6,694868.0,688330.0,3,2026-03-28 02:34:12.648
163475735682498560,163475734680047616,163475735720235008,163475734583574528,163475735741202432,791.991028,758.450989,-1,28,1.886002e6,686785.0,676277.0,3,2026-03-28 02:34:12.710
163475736177426432,163475735720235008,163475736169025536,163475735565041664,163475736206770176,795.036987,758.450989,1,14,973916.0,674989.0,692376.0,3,2026-03-28 02:34:12.892
163475736252923904,163475736169025536,163475736592650240,163475736089329664,163475736684920832,795.036987,767.580017,-1,10,708346.0,688378.0,682272.0,3,2026-03-28 02:34:12.984
163475736835932160,163475736592650240,163475737318264832,163475736370348032,163475737351815168,798.085022,767.580017,1,13,958942.0,689789.0,700481.0,3,2026-03-28 02:34:13.209
…,…,…,…,…,…,…,…,…,…,…,…,…,…
163475749880217600,163475748785491968,163475749863428096,163475748605132800,163475749905367040,823.898987,796.780029,-1,14,937255.0,689741.0,695821.0,3,2026-03-28 02:34:16.181
163475749989269504,163475749863428096,163475750442242048,163475749590794240,163475750534512640,813.625977,796.780029,1,11,683010.0,697376.0,696289.0,3,2026-03-28 02:34:16.281
163475750769410048,163475750442242048,163475751419514880,163475750354157568,163475751461453824,813.625977,790.231018,-1,14,840110.0,700773.0,677202.0,3,2026-03-28 02:34:16.487


In [ ]:

# from pulao.swing import Swing



df = swing_manager.df_swing
#Swing(**df.row(0, named=True))
display(df)
display(sbar_list[0])
print(df.select(pl.col("created_at").cast(pl.Int64)))
print(cbar_manager.df_cbar.select(pl.col("created_at").cast(pl.Int64)))


In [ ]:
# 测试swing 构建算法[查看关联数据]
from IPython.display import display
import logging

from pulao.logging import init_logging

import pandas as pd

from pulao.bar import SBarManager,CBar,CBarManager
from pulao.swing import SwingManager

import polars as pl

from pulao.constant import EventType, Exchange, FractalType, Timeframe,Direction

from pulao.logging import get_logger
init_logging(level=logging.DEBUG)
open("./logs/pulao.log", "w").close() # 每次运行前清空日志
logger = get_logger(__name__)
logger.debug("开始运行")

def format_df_swing():
    # 在df_swing中显示df_cbar的索引顺序
    if "index" in cbar_manager.df_cbar.columns:
        cbar_manager.df_cbar = cbar_manager.df_cbar.drop("index")
    cbar_manager.df_cbar = cbar_manager.df_cbar.with_row_index("index")

    df_cbar = cbar_manager.df_cbar.select(["id","index"])
    df1 = swing_manager.df_swing.join(df_cbar, left_on="cbar_start_id", right_on="id", how="left")
    df2 = swing_manager.df_swing.join(df_cbar, left_on="cbar_end_id", right_on="id", how="left")
    df1 = df1.rename({"index":"cbar_start_index"})
    df2 = df2.rename({"index":"cbar_end_index"})
    df2 = df2.select(["id","cbar_end_index"])
    df1 = df1.join(df2, on="id", how="left")
    df = df1.select(["id","cbar_start_index","cbar_end_index","cbar_start_id","cbar_end_id","high_price","low_price","direction","state","created_at"])

    return df

cbar_manager = CBarManager(sbar_manager=SBarManager(symbol=sbar_manager.symbol, timeframe=cbar_manager.timeframe))
swing_manager = SwingManager(cbar_manager=cbar_manager)
df_cbar = cbar_manager.read_parquet()
cbar_manager.df_cbar = cbar_manager.df_cbar.slice(0,0)
for i in range(df_cbar.height-1):
    cbar = CBar(**df_cbar.row(i, named=True))
    cbar_manager._append_cbar(start_id=cbar.sbar_start_id, end_id=cbar.sbar_end_id, high_price=cbar.high_price, low_price=cbar.low_price, fractal_type=cbar.fractal_type)
    cbar_manager.notify(Timeframe.H1, EventType.CBAR_CHANGED)

logger.debug("运行结束")


display("df_swing")
df = format_df_swing()
display(df)

display("df_cbar")
cbar_manager.df_cbar



In [ ]:
# 可视化日志
import pandas as pd
df = pd.read_json("./logs/pulao.log",lines=True)
df = df.drop(columns=["logger","level","time"])
priority_cols = ["event"]
other_cols = [c for c in df.columns if c not in priority_cols]
df = df[priority_cols+other_cols]
df["info"] = df[other_cols].apply(lambda row: row.dropna().tolist(), axis=1)
df = df.drop(columns=other_cols)
df["info"] = df["info"].apply(lambda x: "" if x == [] else x)
df

In [ ]:
df = sbar_manager.df_sbar
df = df.with_columns(
    pl.concat_list(["ema_short","ema_long"]).alias("ema")
)
df.lazy().select(pl.col("ema")).explain()